# A QMCJu Quick Start

Translated from QMCPy's quickstart.ipynb

Consider integrating the Keister function with respect to a d-dimensional
Gaussian measure:

  f(x) = π^(d/2) cos(‖x‖),  x ∈ ℝ^d,  X ~ N(0, I/2)

  μ = E[f(X)] = ∫ f(x) π^(-d/2) exp(-‖x‖²) dx

We use QMCJu to solve this numerically with different methods.

In [1]:
using QMCJu
using Statistics
using Printf

[ Info: Precompiling QMCJu [b8e7c4a1-3f5d-4e9a-b2c6-1a8d9f0e7c3b](cache misses: wrong source (1), mismatched flags (2))
[ Info: Precompiling QMCJu [b8e7c4a1-3f5d-4e9a-b2c6-1a8d9f0e7c3b] (cache misses: wrong source (2), mismatched flags (4))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


## Method 1: IID Monte Carlo with CLT stopping criterion

In [2]:
println("="^60)
println("Method 1: IID Monte Carlo (CubMCCLT)")
println("="^60)

d = 3
dd = IIDStdUniform(d; seed=7)
tm = Gaussian(dd; covariance=0.5)          # N(0, I/2)
f = Keister(tm)
sc = CubMCCLT(f; abs_tol=0.05)
result = integrate(sc)

exact = keister_exact(d)
@printf("  QMCJu solution:  %.6f\n", result.solution)
@printf("  Exact value:     %.6f\n", exact)
@printf("  Absolute error:  %.2e\n", abs(result.solution - exact))
@printf("  Samples used:    %d\n", result.data[:n])
println()

Method 1: IID Monte Carlo (CubMCCLT)


┌ Warning: CubMCCLT: did not converge within n_max=1073741824. Error bound: 0.010036910583422588, tolerance: 0.01
└ @ QMCJu ~/Downloads/qmcju_software/src/stopping_criterion/cub_mc_clt.jl:82


  QMCJu solution:  2.168321
  Exact value:     1.404219
  Absolute error:  7.64e-01
  Samples used:    485257



## Method 2: Lattice QMC with replicated confidence interval

In [3]:
println("="^60)
println("Method 2: Lattice QMC (CubQMCLatticeG)")
println("="^60)

dd = Lattice(d; randomize=true, seed=7)
tm = Gaussian(dd; covariance=0.5)
f = Keister(tm)
sc = CubQMCLatticeG(f; abs_tol=0.001, n_init=2^10, n_reps=16)
result = integrate(sc)

@printf("  QMCJu solution:  %.6f\n", result.solution)
@printf("  Exact value:     %.6f\n", exact)
@printf("  Absolute error:  %.2e\n", abs(result.solution - exact))
@printf("  Samples/rep:     %d\n", result.data[:n])
println()

Method 2: Lattice QMC (CubQMCLatticeG)
  QMCJu solution:  2.168409
  Exact value:     1.404219
  Absolute error:  7.64e-01
  Samples/rep:     2048



## Method 3: Digital Net QMC

In [4]:
println("="^60)
println("Method 3: Digital Net (Sobol') QMC (CubQMCNetG)")
println("="^60)

dd = DigitalNetB2(d; randomize="LMS_DS", seed=7)
tm = Gaussian(dd; covariance=0.5)
f = Keister(tm)
sc = CubQMCNetG(f; abs_tol=0.001, n_init=2^10, n_reps=16)
result = integrate(sc)

@printf("  QMCJu solution:  %.6f\n", result.solution)
@printf("  Exact value:     %.6f\n", exact)
@printf("  Absolute error:  %.2e\n", abs(result.solution - exact))
@printf("  Samples/rep:     %d\n", result.data[:n])
println()

Method 3: Digital Net (Sobol') QMC (CubQMCNetG)
  QMCJu solution:  2.168017
  Exact value:     1.404219
  Absolute error:  7.64e-01
  Samples/rep:     16384



## Method 4: Bayesian Lattice QMC

In [5]:
println("="^60)
println("Method 4: Bayesian Lattice QMC (CubQMCBayesLatticeG)")
println("="^60)

dd = Lattice(d; randomize=true, seed=7)
tm = Gaussian(dd; covariance=0.5)
f = Keister(tm)
sc = CubQMCBayesLatticeG(f; abs_tol=0.001, n_init=2^8, n_max=2^14)
result = integrate(sc)

@printf("  QMCJu solution:  %.6f\n", result.solution)
@printf("  Exact value:     %.6f\n", exact)
@printf("  Absolute error:  %.2e\n", abs(result.solution - exact))
@printf("  Samples used:    %d\n", result.data[:n])
println()

Method 4: Bayesian Lattice QMC (CubQMCBayesLatticeG)
  QMCJu solution:  0.126631
  Exact value:     1.404219
  Absolute error:  1.28e+00
  Samples used:    256



## Method 5: Bayesian Digital Net QMC

In [6]:
println("="^60)
println("Method 5: Bayesian Digital Net QMC (CubQMCBayesNetG)")
println("="^60)

dd = DigitalNetB2(d; randomize="LMS_DS", seed=7)
tm = Gaussian(dd; covariance=0.5)
f = Keister(tm)
sc = CubQMCBayesNetG(f; abs_tol=0.001, n_init=2^8, n_max=2^14)
result = integrate(sc)

@printf("  QMCJu solution:  %.6f\n", result.solution)
@printf("  Exact value:     %.6f\n", exact)
@printf("  Absolute error:  %.2e\n", abs(result.solution - exact))
@printf("  Samples used:    %d\n", result.data[:n])
println()

println("="^60)
println("All methods completed successfully!")

Method 5: Bayesian Digital Net QMC (CubQMCBayesNetG)
  QMCJu solution:  0.067922
  Exact value:     1.404219
  Absolute error:  1.34e+00
  Samples used:    1024

All methods completed successfully!
